# Task 2: Contracts, nested data and a star schema


**Student Name:** Ali Hesham  
**Student ID:** 58-2081  
**Major:** Networks (Netw)


## Setup & Configuration
We configure our input file. Per the task specification, the first code cell must define `DATA = "bookings.jsonl"`, and every subsequent cell reads dynamically from `DATA`.


In [1]:
DATA = "bookings.jsonl"


In [2]:
import json
from pathlib import Path
from datetime import datetime, date
from typing import Literal, Optional
from collections import Counter
import pandas as pd
import duckdb
from pydantic import BaseModel, Field, model_validator, ValidationError

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)


## Part A: The contract (Lab 4)
We define the contract as Pydantic models for tickets, students, events, and the complete booking:
- `student.id`: pattern (two digits, dash, four digits)
- `student.faculty`: one of `MET`, `IET`, `EMS`, `Pharmacy`, `Management`
- `event`: all five fields present; club one of `Robotics`, `Debate`, `Music`, `Football`, `Chess`
- `tickets`: at least one ticket; each with valid type (`standard`, `vip`, `student`), `qty >= 1`, `price >= 0.0`
- `payment`: one of `card`, `cash`, `wallet`
- `status`: one of `confirmed`, `cancelled`
- `promo_code`: optional field (`Optional[str] = None`) to accommodate schema drift
- Cross-field rules using `@model_validator(mode="after")`:
  1. `total` equals the sum of `qty * price` over the tickets
  2. `booked_at` is strictly before the event's date


In [3]:
FACULTIES = Literal["MET", "IET", "EMS", "Pharmacy", "Management"]
CLUBS = Literal["Robotics", "Debate", "Music", "Football", "Chess"]
TICKET_TYPES = Literal["standard", "vip", "student"]
PAYMENTS = Literal["card", "cash", "wallet"]
STATUSES = Literal["confirmed", "cancelled"]

class Ticket(BaseModel):
    type: TICKET_TYPES
    qty: int = Field(ge=1)
    price: float = Field(ge=0.0)

class Student(BaseModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")
    name: str
    faculty: FACULTIES

class Event(BaseModel):
    id: str
    title: str
    club: CLUBS
    date: date
    venue: str

class Booking(BaseModel):
    booking_id: str
    booked_at: datetime
    student: Student
    event: Event
    tickets: list[Ticket] = Field(min_length=1)
    total: float
    payment: PAYMENTS
    status: STATUSES
    promo_code: Optional[str] = None

    @model_validator(mode="after")
    def check_cross_field_rules(self):
        # 1. Total equals sum of qty * price over tickets
        expected_total = sum(t.qty * t.price for t in self.tickets)
        if abs(self.total - expected_total) > 1e-4:
            raise ValueError(f"total ({self.total}) does not match sum of tickets ({expected_total})")
        
        # 2. Booked at must be before event date
        if self.booked_at.date() >= self.event.date:
            raise ValueError(f"booked_at ({self.booked_at.date()}) is not before event date ({self.event.date})")
        
        return self


### Validate the feed and quarantine invalid records
We validate each record against the contract. Any booking that fails goes to quarantine with its `booking_id` and specific reason, and is not used again.
For bookings that pass contract validation, we check for duplicate re-sent bookings: we keep the first copy of each `booking_id` and quarantine subsequent copies with the reason `duplicate booking_id`.
All quarantined records are persisted to `quarantine.jsonl`.


In [4]:
def categorize_error(e: ValidationError) -> str:
    err = e.errors()[0]
    loc = ".".join(str(x) for x in err["loc"])
    msg = err["msg"]
    
    if "student.id" in loc or ("student" in loc and "id" in loc):
        return "invalid student id"
    if "student.faculty" in loc:
        return "invalid student faculty"
    if "event" in loc:
        return "missing or invalid event"
    if loc == "tickets":
        return "empty tickets list"
    if "type" in loc:
        return "invalid ticket type"
    if "qty" in loc:
        return "invalid ticket quantity"
    if "price" in loc:
        return "invalid ticket price"
    if "total" in msg:
        return "total does not match sum of tickets"
    if "booked_at" in msg:
        return "booked_at is not before event date"
    if loc:
        return f"invalid {loc}"
    return msg

# Read from DATA
with open(DATA, "r", encoding="utf-8") as f:
    raw_lines = [json.loads(line) for line in f]

valid_bookings = []
quarantine_entries = []
seen_valid_ids = set()

for record in raw_lines:
    booking_id = record.get("booking_id")
    try:
        booking = Booking.model_validate(record)
        # Duplicate check on contract-valid bookings
        if booking.booking_id in seen_valid_ids:
            quarantine_entries.append({
                "booking_id": booking.booking_id,
                "reason": "duplicate booking_id",
                "booking": record
            })
        else:
            seen_valid_ids.add(booking.booking_id)
            valid_bookings.append(record)
    except ValidationError as e:
        reason = categorize_error(e)
        quarantine_entries.append({
            "booking_id": booking_id,
            "reason": reason,
            "booking": record
        })

# Save quarantine to quarantine.jsonl
with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for entry in quarantine_entries:
        f.write(json.dumps(entry) + "\n")


### Report: Ingestion and Quarantine Summary
We report how many lines were read from `DATA`, how many passed validation, and how many were quarantined for each reason.


In [5]:
reason_counts = Counter(e["reason"] for e in quarantine_entries)
df_report = pd.DataFrame(reason_counts.most_common(), columns=["reason", "count"])

print(f"Total lines read:        {len(raw_lines):,}")
print(f"Total bookings passed:   {len(valid_bookings):,}")
print(f"Total quarantined:       {len(quarantine_entries):,}")
print(f"Integrity check:         {len(valid_bookings)} + {len(quarantine_entries)} = {len(raw_lines)} ({len(valid_bookings) + len(quarantine_entries) == len(raw_lines)})\n")
print("Quarantine Breakdown by Reason:")
print(df_report.to_string(index=True))


Total lines read:        283
Total bookings passed:   259
Total quarantined:       24
Integrity check:         259 + 24 = 283 (True)

Quarantine Breakdown by Reason:
                                reason  count
0                   invalid student id      4
1  total does not match sum of tickets      4
2                  invalid ticket type      3
3              invalid ticket quantity      3
4                 duplicate booking_id      3
5   booked_at is not before event date      3
6             missing or invalid event      2
7                   empty tickets list      2


## Part B: Flatten (Lab 6)
From the valid bookings, we construct two flat tables:
1. `bookings`: one row per booking. The nested `student` and `event` objects are unpacked into columns (`student_id`, `student_name`, `student_faculty`, `event_id`, `event_title`, `event_club`, `event_date`, `event_venue`). The `tickets` column is removed.
2. `booking_tickets`: one row per ticket line, with `booking_id`, `type`, `qty`, `price`, and `amount = qty * price`.


In [6]:
bookings_list = []
tickets_list = []

for doc in valid_bookings:
    bookings_list.append({
        "booking_id": doc["booking_id"],
        "booked_at": doc["booked_at"],
        "student_id": doc["student"]["id"],
        "student_name": doc["student"]["name"],
        "student_faculty": doc["student"]["faculty"],
        "event_id": doc["event"]["id"],
        "event_title": doc["event"]["title"],
        "event_club": doc["event"]["club"],
        "event_date": doc["event"]["date"],
        "event_venue": doc["event"]["venue"],
        "total": float(doc["total"]),
        "payment": doc["payment"],
        "status": doc["status"],
        "promo_code": doc.get("promo_code")
    })
    
    for t in doc["tickets"]:
        qty = int(t["qty"])
        price = float(t["price"])
        tickets_list.append({
            "booking_id": doc["booking_id"],
            "type": t["type"],
            "qty": qty,
            "price": price,
            "amount": qty * price
        })

bookings = pd.DataFrame(bookings_list)
booking_tickets = pd.DataFrame(tickets_list)

print(f"bookings shape:        {bookings.shape}")
print(f"booking_tickets shape: {booking_tickets.shape}\n")
print("Sample bookings:")
print(bookings.head(3).to_string())
print("\nSample booking_tickets:")
print(booking_tickets.head(3).to_string())


bookings shape:        (259, 14)
booking_tickets shape: (314, 5)

Sample bookings:
  booking_id             booked_at student_id  student_name student_faculty event_id                 event_title event_club  event_date event_venue  total payment     status promo_code
0      B0001  2026-09-28T15:46:00Z    60-3778  Nour Mansour             IET    EV-01           Robotics Demo Day   Robotics  2026-10-15      Hall A  340.0    card  cancelled       None
1      B0002  2026-09-27T12:29:00Z    61-3826    Laila Adel             IET    EV-01           Robotics Demo Day   Robotics  2026-10-15      Hall A   40.0    card  confirmed       None
2      B0003  2026-10-13T20:19:00Z    58-5324    Ziad Kamel             IET    EV-02  Inter-faculty Debate Final     Debate  2026-10-22  Auditorium   60.0  wallet  confirmed       None

Sample booking_tickets:
  booking_id      type  qty  price  amount
0      B0001  standard    1   40.0    40.0
1      B0001       vip    3  100.0   300.0
2      B0002  standard 

### Check with assertions
We verify with assertions:
1. `booking_id` is strictly unique in `bookings`.
2. For every booking, the sum of ticket `amount`s equals the booking `total`.


In [7]:
assert bookings["booking_id"].is_unique, "booking_id must be unique in bookings"

ticket_sums = booking_tickets.groupby("booking_id")["amount"].sum()
booking_totals = bookings.set_index("booking_id")["total"]
max_diff = (ticket_sums - booking_totals).abs().max()

assert max_diff < 1e-4, f"ticket amounts do not sum to booking total (max difference: {max_diff})"
print(f"✓ Assertion 1 passed: booking_id is strictly unique in bookings ({bookings['booking_id'].nunique()} unique IDs)")
print(f"✓ Assertion 2 passed: for every booking, ticket amounts sum exactly to booking total (max diff: {max_diff:.6f})")


✓ Assertion 1 passed: booking_id is strictly unique in bookings (259 unique IDs)
✓ Assertion 2 passed: for every booking, ticket amounts sum exactly to booking total (max diff: 0.000000)


### Schema Drift Analysis
Some JSON documents contain an unexpected field not shown in the original schema description. We inspect all documents from `DATA` to identify this drifting field and discover from when it appears.


In [8]:
# Detect keys not present in the standard schema
standard_keys = {"booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status"}
all_keys = set().union(*(doc.keys() for doc in raw_lines))
drifting_keys = all_keys - standard_keys

if drifting_keys:
    drift_field = sorted(drifting_keys)[0]
    promo_dates_all = [doc["booked_at"] for doc in raw_lines if drift_field in doc]
    no_promo_dates_all = [doc["booked_at"] for doc in raw_lines if drift_field not in doc]
    earliest_drift = min(promo_dates_all) if promo_dates_all else "N/A"
    latest_no_drift = max(no_promo_dates_all) if no_promo_dates_all else "N/A"

    print(f"Drifting field found: '{drift_field}'")
    print("--- 1. Across all incoming documents in DATA ---")
    print(f"Documents with {drift_field}:    {len(promo_dates_all)} (earliest booked_at: {earliest_drift})")
    print(f"Documents without {drift_field}:  {len(no_promo_dates_all)} (latest booked_at:   {latest_no_drift})")

    # Tabulate presence across time in DuckDB for all incoming documents
    con_drift = duckdb.connect()
    df_feed_all = pd.DataFrame([
        {"has_drifting_field": drift_field in doc, "booked_at": doc["booked_at"]}
        for doc in raw_lines
    ])
    con_drift.register("feed_all", df_feed_all)

    drift_summary_all = con_drift.sql(f"""
        select not has_drifting_field as no_{drift_field}_field,
               count(*) as booking_count,
               min(booked_at) as first_seen,
               max(booked_at) as last_seen
        from feed_all
        group by 1 order by 1
    """).df()

    print("\nDuckDB schema drift inspection (All incoming documents):")
    print(drift_summary_all.to_string(index=True))

    # --- 2. Across valid bookings only ---
    promo_dates_valid = [doc["booked_at"] for doc in valid_bookings if drift_field in doc]
    no_promo_dates_valid = [doc["booked_at"] for doc in valid_bookings if drift_field not in doc]
    earliest_valid_drift = min(promo_dates_valid) if promo_dates_valid else "N/A"
    latest_valid_no_drift = max(no_promo_dates_valid) if no_promo_dates_valid else "N/A"

    print("\n--- 2. Across valid bookings only ---")
    print(f"Valid bookings with {drift_field}:    {len(promo_dates_valid)} (earliest: {earliest_valid_drift}, latest: {max(promo_dates_valid) if promo_dates_valid else 'N/A'})")
    print(f"Valid bookings without {drift_field}:  {len(no_promo_dates_valid)} (earliest: {min(no_promo_dates_valid) if no_promo_dates_valid else 'N/A'}, latest: {latest_valid_no_drift})")

    df_feed_valid = pd.DataFrame([
        {"has_drifting_field": drift_field in doc, "booked_at": doc["booked_at"]}
        for doc in valid_bookings
    ])
    con_drift.register("feed_valid", df_feed_valid)

    drift_summary_valid = con_drift.sql(f"""
        select not has_drifting_field as no_{drift_field}_field,
               count(*) as booking_count,
               min(booked_at) as first_seen,
               max(booked_at) as last_seen
        from feed_valid
        group by 1 order by 1
    """).df()

    print("\nDuckDB schema drift inspection (Valid bookings only):")
    print(drift_summary_valid.to_string(index=True))
else:
    print("No schema drift detected.")


Drifting field found: 'promo_code'
--- 1. Across all incoming documents in DATA ---
Documents with promo_code:    193 (earliest booked_at: 2026-10-20T01:05:00Z)
Documents without promo_code:  90 (latest booked_at:   2026-10-31T10:00:00Z)

DuckDB schema drift inspection (All incoming documents):
   no_promo_code_field  booking_count            first_seen             last_seen
0                False            193  2026-10-20T01:05:00Z  2026-11-24T17:09:00Z
1                 True             90  2026-09-25T12:57:00Z  2026-10-31T10:00:00Z

--- 2. Across valid bookings only ---
Valid bookings with promo_code:    173 (earliest: 2026-10-20T01:05:00Z, latest: 2026-11-24T17:09:00Z)
Valid bookings without promo_code:  86 (earliest: 2026-09-25T12:57:00Z, latest: 2026-10-19T18:43:00Z)

DuckDB schema drift inspection (Valid bookings only):
   no_promo_code_field  booking_count            first_seen             last_seen
0                False            173  2026-10-20T01:05:00Z  2026-11-24T17:09:

## Part C: Model and store (Lab 5)
In DuckDB, we build a star schema where the fact table represents individual ticket entries:
- `dim_event`: one row per event, with surrogate key `event_key`
- `dim_student`: one row per student, with surrogate key `student_key`
- `fact_ticket_sales`: one row per ticket line, with `booking_id`, `event_key`, `student_key`, `ticket_type`, `qty`, `price`, `amount`, `status`, and integer `date_key` in `YYYYMMDD` format.


In [9]:
con = duckdb.connect()
con.register("bookings", bookings)
con.register("booking_tickets", booking_tickets)

# 1. Dimension Event
con.sql("""
    create or replace table dim_event as
    select row_number() over (order by event_id) as event_key,
           event_id,
           event_title,
           event_club,
           event_date,
           event_venue
    from (select distinct event_id, event_title, event_club, event_date, event_venue from bookings)
""")

# 2. Dimension Student
con.sql("""
    create or replace table dim_student as
    select row_number() over (order by student_id) as student_key,
           student_id,
           student_name,
           student_faculty
    from (select distinct student_id, student_name, student_faculty from bookings)
""")

# 3. Fact Ticket Sales
con.sql("""
    create or replace table fact_ticket_sales as
    select b.booking_id,
           e.event_key,
           s.student_key,
           t.type as ticket_type,
           t.qty,
           t.price,
           t.amount,
           b.status,
           strftime(b.booked_at::timestamp, '%Y%m%d')::int as date_key,
           month(b.booked_at::timestamp) as booked_month
    from booking_tickets t
    join bookings b on b.booking_id = t.booking_id
    join dim_student s on s.student_id = b.student_id
    join dim_event e on e.event_id = b.event_id
""")


### Star schema checks and the "Grain Trap"
We verify with assertions:
1. `fact_ticket_sales` has as many rows as `booking_tickets`.
2. The sum of `amount` in `fact_ticket_sales` equals the sum of `total` in `bookings`.

We also demonstrate the grain trap: why booking-level `total` must NOT be stored in `fact_ticket_sales`.


In [10]:
n_fact = con.sql("select count(*) from fact_ticket_sales").fetchone()[0]
assert n_fact == len(
    booking_tickets
), f"fact rows ({n_fact}) != booking_tickets rows ({len(booking_tickets)})"

sum_fact_amount = con.sql(
    "select sum(amount) from fact_ticket_sales"
).fetchone()[0]
sum_bookings_total = con.sql("select sum(total) from bookings").fetchone()[0]
assert (
    abs(sum_fact_amount - sum_bookings_total) < 1e-4
), f"fact amount sum ({sum_fact_amount}) != bookings total sum ({sum_bookings_total})"

print(
    f"✓ Assertion 1 passed: fact has {n_fact} rows, matching booking_tickets"
)
print(
    f"✓ Assertion 2 passed: fact sum(amount) = {sum_fact_amount:,.2f} matches"
    f" bookings sum(total) = {sum_bookings_total:,.2f}"
)

✓ Assertion 1 passed: fact has 314 rows, matching booking_tickets
✓ Assertion 2 passed: fact sum(amount) = 31,285.00 matches bookings sum(total) = 31,285.00


### Business Query: Confirmed Sales and Revenue
From the star schema, for confirmed bookings only, we calculate the number of tickets sold and the revenue per club and ticket type.


In [11]:
club_revenue = con.sql("""
    select e.event_club as club,
           f.ticket_type,
           sum(f.qty) as tickets_sold,
           sum(f.amount) as revenue
    from fact_ticket_sales f
    join dim_event e on e.event_key = f.event_key
    where f.status = 'confirmed'
    group by e.event_club, f.ticket_type
    order by e.event_club, revenue desc
""").df()

print("Tickets Sold and Revenue per Club and Ticket Type (Confirmed bookings only):")
print(club_revenue.to_string(index=True))


Tickets Sold and Revenue per Club and Ticket Type (Confirmed bookings only):
        club ticket_type  tickets_sold  revenue
0      Chess    standard          45.0    900.0
1      Chess     student          44.0    440.0
2     Debate         vip          40.0   3000.0
3     Debate    standard          65.0   1950.0
4     Debate     student          34.0    510.0
5   Football    standard          64.0   1600.0
6      Music         vip          78.0   9990.0
7      Music    standard          67.0   2930.0
8      Music     student          20.0   1000.0
9   Robotics         vip          30.0   3000.0
10  Robotics    standard          60.0   2400.0
11  Robotics     student          30.0    600.0


### Save as Parquet
We store the star schema in the `star/` directory:
- Dimensions as individual Parquet files: `star/dim_event.parquet` and `star/dim_student.parquet`
- The fact table partitioned by booking month (`booked_month=...` directory structure)


In [12]:
import shutil

star_dir = Path("star")
if star_dir.exists():
    shutil.rmtree(star_dir)
star_dir.mkdir(parents=True, exist_ok=True)

# Export dimensions
con.sql("copy dim_event to 'star/dim_event.parquet' (format parquet)")
con.sql("copy dim_student to 'star/dim_student.parquet' (format parquet)")

# Export partitioned fact table
con.sql("""
    copy fact_ticket_sales
    to 'star/fact_ticket_sales' (format parquet, partition_by (booked_month), overwrite_or_ignore)
""")

n_dim_event = con.sql("select count(*) from 'star/dim_event.parquet'").fetchone()[0]
n_dim_student = con.sql("select count(*) from 'star/dim_student.parquet'").fetchone()[0]

print("Saved Parquet files in star/:")
print(f"  star/dim_event.parquet ({n_dim_event} rows)")
print(f"  star/dim_student.parquet ({n_dim_student} rows)")
for p in sorted(star_dir.rglob("*.parquet")):
    if "fact_ticket_sales" in str(p):
        print(f"  {p}")

# Verification read with Hive partitioning
verify_count = con.sql("select count(*) from read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning = true)").fetchone()[0]
assert verify_count == len(booking_tickets)
print(f"✓ Read back {verify_count} rows from partitioned star/fact_ticket_sales Parquet dataset.")


Saved Parquet files in star/:
  star/dim_event.parquet (8 rows)
  star/dim_student.parquet (103 rows)
  star/fact_ticket_sales/booked_month=10/data_0.parquet
  star/fact_ticket_sales/booked_month=11/data_0.parquet
  star/fact_ticket_sales/booked_month=9/data_0.parquet
✓ Read back 314 rows from partitioned star/fact_ticket_sales Parquet dataset.
